# R05 - Political Reference Extraction with Mistral

Identify explicit or unambiguous references to political figures, parties, slogans, events, and institutions in French rap lyrics. The current run uses a deterministic sample of exactly 200 songs. Review annotation quality and failures before scaling to the full corpus.

## Outline
1. Configure the job and inspect GPU availability.
2. Start and verify a dedicated local vLLM endpoint.
3. Load the corpus and prepare the 200-song sample.
4. Review the prompt and annotate songs.
5. Inspect references and export results.
6. Calculate reference frequencies by type and canonical reference.

## 1. Configuration and GPU availability

Use exactly 200 songs with a fixed random seed. The GPU check reports all devices and selects the least-used GPU with sufficient free memory and low utilization; it does not assume that GPU 2 is available.

In [ ]:
from pathlib import Path
import importlib
import os
import subprocess
import sys
import time

import pandas as pd
import requests

project_dir = Path.cwd()
sys.path.insert(0, str(project_dir))
import R05_political_references_mistral_job as job
importlib.reload(job)

model_name = 'mistralai/Mistral-7B-Instruct-v0.3'
server_port = 8015
base_url = f'http://127.0.0.1:{server_port}'
corpus_path = Path('corpus.csv')
export_dir = Path('export')
export_dir.mkdir(exist_ok=True)

run_mode = 'test'  # Fixed-size sample; use 'full' only after reviewing annotation quality.
test_sample_size = 200
pilot_fraction = 0.01
random_state = 42
min_words = 10
max_words = 4000
batch_size = 16
request_timeout_seconds = 180
max_retries = 2
minimum_free_gib = 20
max_gpu_utilization = 20
gpu_memory_utilization = 0.35
max_model_len = 8192

def read_gpu_status():
    query = 'index,name,memory.used,memory.total,utilization.gpu'
    result = subprocess.run(
        ['nvidia-smi', f'--query-gpu={query}', '--format=csv,noheader,nounits'],
        check=True, capture_output=True, text=True,
    )
    devices = []
    for line in result.stdout.strip().splitlines():
        index, name, used, total, utilization = [part.strip() for part in line.split(',')]
        devices.append({
            'index': int(index), 'name': name,
            'used_mib': int(used), 'total_mib': int(total),
            'free_gib': (int(total) - int(used)) / 1024,
            'utilization': int(utilization),
        })
    return devices

gpu_status_before = read_gpu_status()
display(pd.DataFrame(gpu_status_before))
gpu_candidates = [
    device for device in gpu_status_before
    if device['free_gib'] >= max(
        minimum_free_gib,
        device['total_mib'] / 1024 * gpu_memory_utilization + 2,
    )
    and device['utilization'] <= max_gpu_utilization
]
if not gpu_candidates:
    raise RuntimeError('No GPU meets the configured free-memory and utilization limits.')
gpu_candidates.sort(key=lambda device: (device['utilization'], -device['free_gib']))
gpu_index = gpu_candidates[0]['index']
print(f"Selected GPU {gpu_index}: {gpu_candidates[0]['name']}, {gpu_candidates[0]['free_gib']:.1f} GiB free")
print(f'Model: {model_name} | vLLM port: {server_port} | run mode: {run_mode}')

## 2. Start and verify the dedicated vLLM endpoint

Use a separate port and bind the server to the GPU selected above. The cell fails explicitly if vLLM cannot start, the requested model is not served, or the server does not appear to allocate GPU memory.

In [ ]:
models_url = f'{base_url}/v1/models'
server_log_path = export_dir / 'R05_political_references_vllm_server.log'
server_process = None

def endpoint_is_healthy():
    try:
        response = requests.get(models_url, timeout=5)
        return response.ok
    except requests.RequestException:
        return False

if endpoint_is_healthy():
    job.check_endpoint(base_url, model_name)
    print(f'Existing endpoint is healthy: {base_url}')
else:
    home = Path.home()
    model_cache = home / '.cache/huggingface/hub/models--mistralai--Mistral-7B-Instruct-v0.3/snapshots'
    snapshots = sorted(model_cache.glob('*')) if model_cache.exists() else []
    model_path = snapshots[0] if snapshots else model_name
    vllm_python = home / 'miniconda3/envs/vllm_mistral/bin/python'
    if not vllm_python.exists():
        vllm_python = Path(sys.executable)
    vllm_library_dir = home / 'miniconda3/envs/vllm_mistral/lib/python3.11/site-packages/vllm'
    server_command = [
        str(vllm_python), '-m', 'vllm.entrypoints.openai.api_server',
        '--model', str(model_path), '--served-model-name', model_name,
        '--host', '127.0.0.1', '--port', str(server_port),
        '--tensor-parallel-size', '1', '--dtype', 'bfloat16',
        '--gpu-memory-utilization', str(gpu_memory_utilization),
        '--max-model-len', str(max_model_len), '--no-enable-log-requests',
    ]
    server_environment = {
        **os.environ,
        'CUDA_VISIBLE_DEVICES': str(gpu_index),
    }
    vllm_env_lib_dir = vllm_python.parent.parent / 'lib'
    compatible_libstdcpp = vllm_env_lib_dir / 'libstdc++.so.6'
    library_paths = [str(vllm_env_lib_dir), str(vllm_library_dir)]
    library_paths = [path for path in library_paths if Path(path).exists()]
    server_environment['LD_LIBRARY_PATH'] = ':'.join(
        library_paths + [server_environment.get('LD_LIBRARY_PATH', '')]
    )
    if compatible_libstdcpp.exists():
        server_environment['LD_PRELOAD'] = str(compatible_libstdcpp)
    with server_log_path.open('a', encoding='utf-8') as server_log:
        server_process = subprocess.Popen(
            server_command, stdout=server_log, stderr=subprocess.STDOUT,
            env=server_environment,
        )
    deadline = time.monotonic() + 900
    while time.monotonic() < deadline:
        if endpoint_is_healthy():
            break
        if server_process.poll() is not None:
            raise RuntimeError(f'vLLM exited; inspect {server_log_path}')
        time.sleep(5)
    else:
        raise TimeoutError(f'vLLM did not start within 15 minutes; inspect {server_log_path}')
    job.check_endpoint(base_url, model_name)
    gpu_status_after = read_gpu_status()
    selected_after = next(device for device in gpu_status_after if device['index'] == gpu_index)
    selected_before = next(device for device in gpu_status_before if device['index'] == gpu_index)
    allocated_gib = (selected_after['used_mib'] - selected_before['used_mib']) / 1024
    if allocated_gib < 4:
        raise RuntimeError(
            f'Endpoint is healthy but GPU {gpu_index} allocation increased by only '
            f'{allocated_gib:.1f} GiB; confirm model placement before annotation.'
        )
    print(f'vLLM is healthy on GPU {gpu_index}; observed allocation increase: {allocated_gib:.1f} GiB')

print(f'Endpoint/model verified: {base_url} | {model_name}')

## 3. Load and sample the corpus

Filter to songs with 10–4,000 words. Select exactly 200 songs with a fixed random seed; output filenames include `200songs_10to4000w` to preserve earlier results.

In [ ]:
if not corpus_path.exists():
    raise FileNotFoundError(f'Corpus not found: {corpus_path.resolve()}')
sample_df, sample_label = job.load_corpus(
    corpus_path, mode=run_mode, pilot_fraction=pilot_fraction,
    random_state=random_state, min_words=min_words, max_words=max_words,
    sample_size=test_sample_size,
)
print(f'{len(sample_df):,} songs selected ({sample_label})')
display(sample_df[['doc_id', 'artist', 'title', 'year', 'word_count']].head())

## 4. Review the annotation prompt

The model returns only a JSON list. Each item is validated against the five-category schema and must include an exact mention and a short evidence excerpt found in the lyrics.

In [ ]:
print(job.SYSTEM_PROMPT)
print('\nExample user message:')
print(job.build_user_prompt('J’ai entendu parler de Macron à l’Assemblée nationale.'))
job.check_endpoint(base_url, model_name)

## 5. Annotate, inspect, and export

The progress bar reports completed songs, references, failures, and throughput. The primary JSON preserves one record per song; failed annotations are explicitly marked and also written to a diagnostics CSV rather than being silently treated as songs with no references.

In [ ]:
started_at = time.monotonic()
results = job.annotate_corpus(
    sample_df, base_url=base_url, model_name=model_name,
    timeout=request_timeout_seconds, max_retries=max_retries,
    batch_size=batch_size,
)
elapsed_seconds = time.monotonic() - started_at
json_path, csv_path, errors_path = job.export_results(results, export_dir, sample_label)
annotation_metrics_path = job.export_annotation_metrics(results, export_dir, sample_label, elapsed_seconds)
reference_count = sum(len(song['references']) for song in results)
failure_count = sum(song['annotation_status'] == 'failed' for song in results)
print(f'Processed {len(results):,} songs, found {reference_count:,} references, and recorded {failure_count:,} failed annotations.')
print(f'Elapsed: {elapsed_seconds:.1f} seconds; throughput: {len(results) / max(elapsed_seconds, 1e-9):.2f} songs/s')
print(f'JSON: {json_path}\nCSV: {csv_path}\nDiagnostics: {errors_path}')
print(f'Annotation performance: {annotation_metrics_path}')
assert len(results) == len(sample_df)

### 5.1 Review the test output

Inspect examples, category counts, and failed annotations in the 200-song sample before increasing the run scope.

In [ ]:
reference_rows = [
    {'doc_id': song['doc_id'], 'artist': song['artist'], 'title': song['title'],
     'year': song['year'], **reference}
    for song in results for reference in song['references']
]
references_df = pd.DataFrame(
    reference_rows,
    columns=['doc_id', 'artist', 'title', 'year', 'type', 'mention', 'reference', 'extrait'],
)
if references_df.empty:
    print('No political references were extracted in this sample.')
else:
    display(references_df.head(20))
    display(references_df['type'].value_counts().rename_axis('type').to_frame('count'))
gpu_status_after_annotation = read_gpu_status()
display(pd.DataFrame(gpu_status_after_annotation))
selected_gpu_after = next(device for device in gpu_status_after_annotation if device['index'] == gpu_index)
print(
    f"Selected GPU {gpu_index} ({selected_gpu_after['name']}): "
    f"{selected_gpu_after['used_mib']:,} MiB used / "
    f"{selected_gpu_after['total_mib']:,} MiB total, "
    f"{selected_gpu_after['utilization']}% utilization after annotation."
)

### 5.2 Build reference_canon from Mistral reference labels

`reference` is the original label produced by Mistral and must remain unchanged. `reference_canon` is a separate column constructed by our embedding-based reconciliation process. Compute local BGE-M3 embeddings for distinct reference labels, then resolve aliases within each reference type. Use a cosine threshold of 0.85 and a minimum best-match margin of 0.03. Prefer explicit canonical names, then fuller and more frequent labels. `Emmanuel Macron` is an editable preferred label so that `Macron`, `Emmanuel Macron`, and the typo `Président Marcon` can share that canonical name.

For people, remove titles during name comparison, correct only unambiguous near-spellings of known full-name tokens, and reject matches with conflicting given names. Embeddings compare direct representatives rather than transitive chains. Ambiguous labels are retained. These safeguards are conservative, not proof of identity: review the exported alias mapping and cosine scores. In particular, unrelated people can have similar embeddings.

Run the cached model on CPU to avoid interfering with Mistral. The kernel needs compatible PyTorch and Transformers installations (the local `vllm_mistral` environment was validated). Preview only the first 10 rows of the enriched table and alias mapping; save complete CSVs and performance metrics, preserving raw annotations.

In [ ]:
importlib.reload(job)

canonical_similarity_threshold = 0.85
canonical_ambiguity_margin = 0.03
canonical_embedding_batch_size = 16
preferred_canonical_names = {'figure_politique': ['Emmanuel Macron']}

references_df, canonical_mapping_df, canonical_metrics = job.canonicalize_references(
    references_df, preferred_names=preferred_canonical_names,
    similarity_threshold=canonical_similarity_threshold,
    ambiguity_margin=canonical_ambiguity_margin,
    embedding_batch_size=canonical_embedding_batch_size,
)
canonical_lookup = {
    (row['type'], row['reference']): row['reference_canon']
    for row in references_df[['type', 'reference', 'reference_canon']].drop_duplicates().to_dict('records')
}
canonical_references_path = export_dir / f'R05_political_references_{sample_label}_canonical.csv'
canonical_mapping_path = export_dir / f'R05_political_references_{sample_label}_canonical_mapping.csv'
canonical_metrics_path = export_dir / f'R05_political_references_{sample_label}_canonical_performance.csv'
references_df.to_csv(canonical_references_path, index=False, encoding='utf-8')
canonical_mapping_df.to_csv(canonical_mapping_path, index=False, encoding='utf-8')
pd.DataFrame([canonical_metrics]).to_csv(canonical_metrics_path, index=False, encoding='utf-8')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(references_df.head(10))
    display(canonical_mapping_df.head(10))
print(f'References with canonical labels: {canonical_references_path}')
print(f'Alias mapping: {canonical_mapping_path}')
print(f'Canonicalization performance: {canonical_metrics_path}')

## 6. Reference frequencies by type

For each of the five categories, calculate the number of extracted references, its percentage of all references, the number of songs containing that type, and its prevalence among successfully annotated songs. A reference is an extracted item, not a count of repeated lyric occurrences. Each song counts at most once per type, but can contain several types; song percentages therefore need not sum to 100%. Failed annotations are excluded from the song denominator. Categories with no references remain visible. Reference shares are undefined when no references were extracted.

Export the table to `export/` with the current sample label.

In [ ]:
successful_results = [song for song in results if song['annotation_status'] == 'success']
successful_song_count = len(successful_results)
if successful_song_count == 0:
    raise ValueError('No successful annotations are available for frequency analysis.')

reference_counts = {reference_type: 0 for reference_type in sorted(job.REFERENCE_TYPES)}
song_counts = dict.fromkeys(reference_counts, 0)
for song in successful_results:
    song_types = set()
    for reference in song['references']:
        reference_counts[reference['type']] += 1
        song_types.add(reference['type'])
    for reference_type in song_types:
        song_counts[reference_type] += 1

total_references = sum(reference_counts.values())
type_frequency_df = pd.DataFrame([
    {
        'type': reference_type,
        'reference_count': count,
        'reference_share_pct': 100 * count / total_references if total_references else pd.NA,
        'song_count': song_counts[reference_type],
        'successful_song_count': successful_song_count,
        'song_prevalence_pct': 100 * song_counts[reference_type] / successful_song_count,
    }
    for reference_type, count in reference_counts.items()
]).sort_values(['reference_count', 'type'], ascending=[False, True]).reset_index(drop=True)

type_frequency_path = export_dir / f'R05_political_references_{sample_label}_type_frequencies.csv'
type_frequency_df.to_csv(type_frequency_path, index=False, encoding='utf-8')
display(type_frequency_df.loc[type_frequency_df['reference_count'] > 1].head(10))
print(f'Frequency analysis: {total_references:,} references in {successful_song_count:,} successfully annotated songs.')
print(f'Excluded failed annotations: {len(results) - successful_song_count:,}')
print(f'Type frequencies: {type_frequency_path}')

### 6.1 Frequencies by type and canonical reference

Group extracted items by category and `reference_canon`, constructed by our embedding-based process in section 5.2, not by the original Mistral `reference` label. Show the extraction count, share within that category, number of distinct songs containing the reference, and prevalence among successfully annotated songs. Several mentions of the same canonical reference in one song contribute several extracted items but only one song. Failed annotations are excluded.

Preview at most the first 10 rows with `reference_count > 1` (at least two extractions). Keep all rows in the sample-labeled CSV and calculate percentages before this display-only filter, without rerunning inference.

In [ ]:
successful_reference_songs = [song for song in results if song['annotation_status'] == 'success']
reference_song_denominator = len(successful_reference_songs)
if reference_song_denominator == 0:
    raise ValueError('No successful annotations are available for reference frequency analysis.')

reference_frequency_rows = [
    {'song_index': song_index, 'type': reference['type'],
     'reference_canon': canonical_lookup[(reference['type'], reference['reference'])]}
    for song_index, song in enumerate(successful_reference_songs)
    for reference in song['references']
]
reference_frequency_columns = [
    'type', 'reference_canon', 'reference_count', 'reference_share_within_type_pct',
    'song_count', 'successful_song_count', 'song_prevalence_pct',
]
if reference_frequency_rows:
    reference_frequency_df = (
        pd.DataFrame(reference_frequency_rows)
        .groupby(['type', 'reference_canon'], as_index=False)
        .agg(reference_count=('song_index', 'size'), song_count=('song_index', 'nunique'))
    )
    type_totals = reference_frequency_df.groupby('type')['reference_count'].transform('sum')
    reference_frequency_df['reference_share_within_type_pct'] = (
        100 * reference_frequency_df['reference_count'] / type_totals
    )
    reference_frequency_df['successful_song_count'] = reference_song_denominator
    reference_frequency_df['song_prevalence_pct'] = (
        100 * reference_frequency_df['song_count'] / reference_song_denominator
    )
    reference_frequency_df = reference_frequency_df[reference_frequency_columns].sort_values(
        ['type', 'reference_count', 'reference_canon'], ascending=[True, False, True],
    ).reset_index(drop=True)
else:
    reference_frequency_df = pd.DataFrame(columns=reference_frequency_columns)
    print('No political references were extracted in the successful annotations.')

reference_frequency_path = export_dir / f'R05_political_references_{sample_label}_type_reference_frequencies.csv'
reference_frequency_df.to_csv(reference_frequency_path, index=False, encoding='utf-8')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(reference_frequency_df.loc[reference_frequency_df['reference_count'] > 1].head(10))
print(f'Type/reference frequencies: {reference_frequency_path}')